# Kaggle NLP TF-IDF Template — 5 Hour Selection / No Internet

Template untuk text classification/regression tanpa LLM, internet, atau pretrained transformer.

Fitur:
- auto-discover train/test/sample submission
- auto-detect target/id/text columns
- TF-IDF word ngram + char ngram
- Logistic Regression / Naive Bayes / LinearSVC calibrated / Ridge
- CV + submission generator

Catatan umum:
- Template ini sengaja generic karena detail kompetisi belum diketahui.
- Auto-detect bisa salah. Bagian paling penting adalah cell `CONFIG`.
- Selalu cek `sample_submission.csv`, metric, dan format kolom sebelum final submit.
- Target pertama saat seleksi: buat `submission.csv` valid secepat mungkin.

In [ ]:
import os, re, gc, glob, math, warnings, random
from pathlib import Path
from pprint import pprint
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy.sparse import hstack

from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, log_loss, mean_squared_error, mean_absolute_error, r2_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

CONFIG = {
    'TRAIN_FILE': None,
    'TEST_FILE': None,
    'SAMPLE_SUBMISSION_FILE': None,
    'TARGET_COL': None,
    'ID_COL': None,
    'TEXT_COLS': None,
    # None, 'binary', 'multiclass', 'regression'
    'TASK_TYPE': None,
    # 'logreg', 'nb', 'linear_svc', 'ridge'
    'MODEL': 'logreg',
    'N_SPLITS': 5,
    'PRIMARY_METRIC': None,
    'WORD_NGRAM': (1, 2),
    'CHAR_NGRAM': (3, 5),
    'MAX_WORD_FEATURES': 200_000,
    'MAX_CHAR_FEATURES': 200_000,
    'MIN_DF': 2,
    'SUBMISSION_FILE': 'submission.csv',
}

In [ ]:
INPUT_ROOT = Path('/kaggle/input')
if not INPUT_ROOT.exists(): INPUT_ROOT = Path('.')

csv_files = sorted(str(p) for p in INPUT_ROOT.glob('**/*.csv'))
print(f'Found {len(csv_files)} CSV files')
for i, f in enumerate(csv_files):
    try:
        prev = pd.read_csv(f, nrows=3)
        print(f'{i:02d}. {f} | columns={list(prev.columns)}')
    except Exception as e:
        print(f'{i:02d}. {f} | cannot preview: {e}')

def pick_file(files, keywords):
    keywords = [k.lower() for k in keywords]
    for f in files:
        if any(k in Path(f).name.lower() for k in keywords):
            return f
    return None

CONFIG['TRAIN_FILE'] = CONFIG['TRAIN_FILE'] or pick_file(csv_files, ['train'])
CONFIG['TEST_FILE'] = CONFIG['TEST_FILE'] or pick_file(csv_files, ['test'])
CONFIG['SAMPLE_SUBMISSION_FILE'] = CONFIG['SAMPLE_SUBMISSION_FILE'] or pick_file(csv_files, ['sample', 'submission'])

pprint({k: CONFIG[k] for k in ['TRAIN_FILE', 'TEST_FILE', 'SAMPLE_SUBMISSION_FILE']})
assert CONFIG['TRAIN_FILE'] is not None and CONFIG['TEST_FILE'] is not None

In [ ]:
train = pd.read_csv(CONFIG['TRAIN_FILE'])
test = pd.read_csv(CONFIG['TEST_FILE'])
sample_submission = pd.read_csv(CONFIG['SAMPLE_SUBMISSION_FILE']) if CONFIG['SAMPLE_SUBMISSION_FILE'] else None
print('train:', train.shape)
print('test :', test.shape)
if sample_submission is not None:
    print('sample_submission:', sample_submission.shape)
    display(sample_submission.head())
display(train.head())
display(test.head())

In [ ]:
def auto_detect_target(train_df, test_df):
    diff = [c for c in train_df.columns if c not in test_df.columns]
    if len(diff) == 1: return diff[0]
    for c in ['target', 'label', 'class', 'sentiment', 'score', 'y']:
        if c in train_df.columns and c not in test_df.columns: return c
    return train_df.columns[-1]

def auto_detect_id(train_df, test_df, sample_df=None):
    common = [c for c in train_df.columns if c in test_df.columns]
    if sample_df is not None and sample_df.columns[0] in test_df.columns:
        return sample_df.columns[0]
    for c in ['id', 'ID', 'Id', 'row_id', 'text_id']:
        if c in common: return c
    if common and train_df[common[0]].is_unique and test_df[common[0]].is_unique:
        return common[0]
    return None

def auto_detect_text_cols(train_df, test_df, target_col, id_col):
    candidates = []
    for c in train_df.columns:
        if c not in test_df.columns or c in [target_col, id_col]: continue
        if train_df[c].dtype == 'object' or test_df[c].dtype == 'object':
            s = pd.concat([train_df[c], test_df[c]], axis=0).dropna().astype(str)
            if len(s) == 0: continue
            avg_len = s.str.len().mean()
            uniq_ratio = s.nunique() / max(len(s), 1)
            name_score = int(any(k in c.lower() for k in ['text', 'comment', 'review', 'title', 'body', 'description', 'excerpt']))
            if avg_len >= 15 or uniq_ratio > 0.2 or name_score:
                candidates.append((c, name_score, avg_len, uniq_ratio))
    candidates = sorted(candidates, key=lambda x: (x[1], x[2], x[3]), reverse=True)
    return [c[0] for c in candidates] if candidates else []

CONFIG['TARGET_COL'] = CONFIG['TARGET_COL'] or auto_detect_target(train, test)
CONFIG['ID_COL'] = CONFIG['ID_COL'] or auto_detect_id(train, test, sample_submission)
CONFIG['TEXT_COLS'] = CONFIG['TEXT_COLS'] or auto_detect_text_cols(train, test, CONFIG['TARGET_COL'], CONFIG['ID_COL'])

target_col = CONFIG['TARGET_COL']; id_col = CONFIG['ID_COL']; text_cols = CONFIG['TEXT_COLS']
print('TARGET_COL:', target_col)
print('ID_COL    :', id_col)
print('TEXT_COLS :', text_cols)
assert len(text_cols) > 0, "Set CONFIG['TEXT_COLS'] manual, contoh ['text']"

In [ ]:
y_raw = train[target_col]
def infer_task_type(y):
    if CONFIG['TASK_TYPE'] is not None: return CONFIG['TASK_TYPE']
    if y.dtype == 'object' or str(y.dtype).startswith('category') or str(y.dtype) == 'bool':
        return 'binary' if y.nunique() <= 2 else 'multiclass'
    if y.nunique() <= 20 and y.nunique() / len(y) < 0.05:
        return 'binary' if y.nunique() <= 2 else 'multiclass'
    return 'regression'

task_type = infer_task_type(y_raw)
print('TASK_TYPE:', task_type)
display(y_raw.value_counts(dropna=False).head(20))

In [ ]:
def clean_text(x):
    x = str(x).replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')
    x = re.sub(r'\s+', ' ', x).strip()
    return x

def build_text(df, cols):
    parts = [df[c].fillna('').astype(str).map(clean_text) for c in cols]
    text = parts[0]
    for p in parts[1:]:
        text = text + ' [SEP] ' + p
    return text

train_text = build_text(train, text_cols)
test_text = build_text(test, text_cols)
print('Sample text:')
for i in range(min(3, len(train_text))):
    print('-' * 80)
    print(train_text.iloc[i][:800])
print('Length stats:')
display(train_text.str.len().describe())

In [ ]:
if task_type in ['binary', 'multiclass']:
    target_encoder = LabelEncoder()
    y = target_encoder.fit_transform(y_raw.astype(str))
    n_classes = len(target_encoder.classes_)
    print('Classes:', list(target_encoder.classes_))
else:
    target_encoder = None
    y = pd.to_numeric(y_raw, errors='coerce').values
    n_classes = None

In [ ]:
word_vectorizer = TfidfVectorizer(
    analyzer='word', ngram_range=CONFIG['WORD_NGRAM'], min_df=CONFIG['MIN_DF'],
    max_features=CONFIG['MAX_WORD_FEATURES'], strip_accents='unicode', sublinear_tf=True
)
char_vectorizer = TfidfVectorizer(
    analyzer='char_wb', ngram_range=CONFIG['CHAR_NGRAM'], min_df=CONFIG['MIN_DF'],
    max_features=CONFIG['MAX_CHAR_FEATURES'], sublinear_tf=True
)

all_text = pd.concat([train_text, test_text], axis=0).astype(str)
print('Fitting word tfidf...')
word_vectorizer.fit(all_text)
print('Fitting char tfidf...')
char_vectorizer.fit(all_text)

X = hstack([word_vectorizer.transform(train_text), char_vectorizer.transform(train_text)]).tocsr()
X_test = hstack([word_vectorizer.transform(test_text), char_vectorizer.transform(test_text)]).tocsr()
print('X:', X.shape, 'X_test:', X_test.shape)

In [ ]:
def build_model():
    name = CONFIG['MODEL']
    if task_type == 'regression':
        return Ridge(alpha=1.0, random_state=RANDOM_STATE)
    if name == 'logreg':
        return LogisticRegression(C=2.0, max_iter=300, solver='saga', n_jobs=-1, class_weight='balanced', random_state=RANDOM_STATE)
    if name == 'nb':
        return ComplementNB(alpha=0.5) if task_type == 'binary' else MultinomialNB(alpha=0.5)
    if name == 'linear_svc':
        return CalibratedClassifierCV(LinearSVC(C=1.0, class_weight='balanced', random_state=RANDOM_STATE), cv=3)
    if name == 'ridge':
        from sklearn.linear_model import RidgeClassifier
        return RidgeClassifier(alpha=1.0, class_weight='balanced', random_state=RANDOM_STATE)
    raise ValueError(name)

def classification_metrics(y_true, proba=None, label=None):
    if label is None and proba is not None: label = np.argmax(proba, axis=1)
    out = {}
    if label is not None:
        out['accuracy'] = accuracy_score(y_true, label)
        out['f1_macro'] = f1_score(y_true, label, average='macro')
    if proba is not None:
        try: out['logloss'] = log_loss(y_true, proba)
        except Exception: pass
        try: out['auc'] = roc_auc_score(y_true, proba[:, 1]) if proba.shape[1] == 2 else roc_auc_score(y_true, proba, multi_class='ovr')
        except Exception: pass
    return out

def regression_metrics(y_true, pred):
    return {'rmse': mean_squared_error(y_true, pred, squared=False), 'mae': mean_absolute_error(y_true, pred), 'r2': r2_score(y_true, pred)}

def show(d, prefix=''):
    print(prefix + ' | '.join(f'{k}: {v:.5f}' for k, v in d.items()))

In [ ]:
if task_type in ['binary', 'multiclass']:
    splitter = StratifiedKFold(n_splits=CONFIG['N_SPLITS'], shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros((X.shape[0], n_classes), dtype=np.float32)
    test_pred = np.zeros((X_test.shape[0], n_classes), dtype=np.float32)
else:
    splitter = KFold(n_splits=CONFIG['N_SPLITS'], shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros(X.shape[0], dtype=np.float32)
    test_pred = np.zeros(X_test.shape[0], dtype=np.float32)

models = []
for fold, (tr_idx, va_idx) in enumerate(splitter.split(X, y if task_type != 'regression' else None), 1):
    print('=' * 80)
    print('Fold', fold)
    model = build_model()
    model.fit(X[tr_idx], y[tr_idx])
    if task_type in ['binary', 'multiclass']:
        if hasattr(model, 'predict_proba'):
            va_proba = model.predict_proba(X[va_idx])
            te_proba = model.predict_proba(X_test)
            va_label = np.argmax(va_proba, axis=1)
        else:
            va_label = model.predict(X[va_idx])
            te_label = model.predict(X_test)
            va_proba = np.eye(n_classes)[va_label]
            te_proba = np.eye(n_classes)[te_label]
        if va_proba.ndim == 1:
            va_proba = np.vstack([1-va_proba, va_proba]).T
            te_proba = np.vstack([1-te_proba, te_proba]).T
        oof[va_idx] = va_proba
        test_pred += te_proba / CONFIG['N_SPLITS']
        show(classification_metrics(y[va_idx], va_proba, va_label), f'Fold {fold}: ')
    else:
        va_pred = model.predict(X[va_idx])
        te_pred = model.predict(X_test)
        oof[va_idx] = va_pred
        test_pred += te_pred / CONFIG['N_SPLITS']
        show(regression_metrics(y[va_idx], va_pred), f'Fold {fold}: ')
    models.append(model)
    gc.collect()

print('OOF')
show(classification_metrics(y, oof) if task_type != 'regression' else regression_metrics(y, oof), 'OOF: ')

In [ ]:
best_threshold = 0.5
if task_type == 'binary':
    probs = oof[:, 1]
    best_f1, best_t_f1 = -1, 0.5
    best_acc, best_t_acc = -1, 0.5
    for t in np.linspace(0.05, 0.95, 181):
        pred = (probs >= t).astype(int)
        f1 = f1_score(y, pred); acc = accuracy_score(y, pred)
        if f1 > best_f1: best_f1, best_t_f1 = f1, t
        if acc > best_acc: best_acc, best_t_acc = acc, t
    print(f'Best F1 threshold={best_t_f1:.3f} f1={best_f1:.5f}')
    print(f'Best Acc threshold={best_t_acc:.3f} acc={best_acc:.5f}')
    if CONFIG['PRIMARY_METRIC'] == 'f1': best_threshold = best_t_f1
    if CONFIG['PRIMARY_METRIC'] == 'accuracy': best_threshold = best_t_acc
print('Selected threshold:', best_threshold)

In [ ]:
def make_submission():
    if sample_submission is not None:
        sub = sample_submission.copy()
        id_candidate = sub.columns[0]
        pred_cols = [c for c in sub.columns if c != id_candidate]
    else:
        sub = pd.DataFrame()
        sub[id_col if id_col is not None else 'id'] = test[id_col].values if id_col is not None else np.arange(len(test))
        pred_cols = ['target']; sub['target'] = 0
    if task_type == 'regression':
        sub[pred_cols[0]] = test_pred
    elif task_type == 'binary':
        if len(pred_cols) == 1:
            sub[pred_cols[0]] = test_pred[:, 1]
            # If label submission is needed:
            # lbl = (test_pred[:, 1] >= best_threshold).astype(int)
            # sub[pred_cols[0]] = target_encoder.inverse_transform(lbl)
        else:
            for i, c in enumerate(pred_cols[:test_pred.shape[1]]): sub[c] = test_pred[:, i]
    else:
        if len(pred_cols) == test_pred.shape[1]:
            for i, c in enumerate(pred_cols): sub[c] = test_pred[:, i]
        else:
            lbl = np.argmax(test_pred, axis=1)
            sub[pred_cols[0]] = target_encoder.inverse_transform(lbl)
    return sub

submission = make_submission()
display(submission.head())
print('shape:', submission.shape, 'NaN:', submission.isna().sum().sum())
submission.to_csv(CONFIG['SUBMISSION_FILE'], index=False)
print('Saved:', CONFIG['SUBMISSION_FILE'])

## NLP quick improvement list

- Coba `CONFIG['MODEL']='nb'` untuk text pendek/noisy.
- Coba `linear_svc` kalau metric F1/accuracy dan dataset tidak terlalu besar.
- Ubah `WORD_NGRAM` ke `(1, 3)` jika masih cepat.
- Untuk binary AUC/logloss, submit probability; untuk F1/accuracy, submit label.